# CLM Native v1 — bounded dynamic sparse paths

Import this notebook into your original **bingjiangli/minicells** Kaggle editor. Keep its existing **GITHUB_TOKEN** secret enabled. No external dataset, Hugging Face token or pretrained model is required.

The default `preflight` inspects resources and exact parameter matching. Set only `MODE = 'screen'` to run the preregistered single-seed twelve-job A/B/C × B0/B1/B2/C0 screening. Two T4 GPUs run independent arm queues concurrently. This does not approve formal GPU confirmation or sealed-data evaluation.

`push_existing` retries GitHub publication of completed output without training. Checkpoints and completed runs are preserved. No manual setup or one-hour timeout. Training stops at the scientific step budget. Do not increase model size or training budget to avoid a negative result.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
MODE = 'preflight'  # preflight | screen | push_existing
SOURCE_COMMIT = '0000000000000000000000000000000000000000'  # pinned implementation
ROOT = Path('/kaggle/working/clm-native-code-' + SOURCE_COMMIT[:12])
RUN_ROOT = Path('/kaggle/working/clm-native-screen-' + SOURCE_COMMIT[:12])
PUBLISH = True
REPOSITORY_URL = 'https://github.com/ArcheLabs/mini-cells.git'


## Load only source and small shared components; no historical weights.

In [ ]:
if MODE not in ('preflight', 'screen', 'push_existing'):
    raise ValueError('Choose preflight, screen, or push_existing')
if len(SOURCE_COMMIT) != 40 or SOURCE_COMMIT == '0' * 40:
    raise ValueError('Import the notebook with its frozen implementation commit')
if not ROOT.exists():
    ROOT.mkdir(parents=True)
    def git(*args):
        subprocess.run(['git', *args], cwd=ROOT, check=True)
    git('init', '--initial-branch=research')
    git('remote', 'add', 'origin', REPOSITORY_URL)
    git('config', 'remote.origin.promisor', 'true')
    git('config', 'remote.origin.partialclonefilter', 'blob:none')
    git('sparse-checkout', 'init', '--no-cone')
    git('sparse-checkout', 'set', '--no-cone',
        '/pyproject.toml', '/src/', '/shared/python/',
        '/research/projects/clm-native/',
        '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/exp005/__init__.py',
        '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/exp005/checkpoint.py',
        '/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/publish_experiment_results.py',
        '/research/projects/continual-learning/context/04-continual-learning/scripts/research/publish_core_validation_007.py')
    git('fetch', '--depth=1', '--filter=blob:none', 'origin', SOURCE_COMMIT)
    git('checkout', '--detach', SOURCE_COMMIT)
import importlib.util
missing = [name for name, module in [('numpy','numpy'),('pyyaml','yaml'),('matplotlib','matplotlib')] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *missing], check=True)
EXP = ROOT / 'research/projects/clm-native/experiments/native-sparse-path'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
def command(mode):
    return [sys.executable, str(EXP/'run.py'), '--mode', mode, '--source-commit', SOURCE_COMMIT, '--output', str(RUN_ROOT)]
subprocess.run(command('preflight'), cwd=ROOT, check=True)


## Verify GitHub write access before any training.

In [ ]:
if MODE in ('screen', 'push_existing') and PUBLISH:
    token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not token:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GITHUB_TOKEN').strip()
    if not token:
        raise RuntimeError('Enable the existing GITHUB_TOKEN secret in bingjiangli/minicells')
    os.environ['GITHUB_TOKEN'] = token
    token = ''
    subprocess.run(command('publish') + ['--preflight-only'], cwd=ROOT, check=True)


## Execute the screening, analyze its gate, and return raw evidence to GitHub.

In [ ]:
if MODE == 'screen':
    import torch
    devices = ['cuda:0', 'cuda:1'] if torch.cuda.device_count() >= 2 else ['cuda:0'] if torch.cuda.is_available() else ['cpu']
    print({'mode': MODE, 'devices': devices, 'source': SOURCE_COMMIT, 'stage': 'SCREENING', 'scientific_decision': False}, flush=True)
    subprocess.run(command('screen') + ['--devices', *devices], cwd=ROOT, check=True)
if MODE in ('screen', 'push_existing') and PUBLISH:
    subprocess.run(command('publish'), cwd=ROOT, check=True)
print({'mode': MODE, 'results': str(RUN_ROOT), 'formal_confirmation': 'NOT_REGISTERED'}, flush=True)
